# ETL Pipeline: Data Kendaraan Listrik (EV)

Milestone 1

Nama  : Narda Parama Agung Noviantoro

Batch : CODA-RMT-0023

File .ipynb ini digunakan untuk melakukan proses Extract dan Transform dari keseluruhan ETL Pipeline pada data spesifikasi kendaraan lsitrik (EV) yang di ambil dara website ev-database.org. 

Proses Extract data akan dilakukan melalui web scraping menggunakan Selenium dan BeautifulSoup.

Proses Transform akan dilakukan dengan Pandas untuk pembersihan dataset.

Proses Load akan dilakukan didalam PostgreSQL.

### A. Extract

Script dibawah menggunakan Selenium dengan niat otomasikan chrome browser untuk capture data HTML di dalam website 'Electric Vehicle Database'.

HTML tersebut akan kemudian di parse menggunakan BeautifulSoup untuk extract data 'Car Model', 'Efficiency (Wh/km)', 'Range (km)', 'Battery (kWh)', 'Price (GBP)', dan 'Availability'.

Data extraction juga menggunakan try/except untuk input 'None' values.

In [26]:
import time
import pandas as pd
from selenium import webdriver
from bs4 import BeautifulSoup

model = []
efficiency = []
range_km = []
battery = []
price = []
availability = []

driver = webdriver.Chrome()

url = 'https://ev-database.org/#group=vehicle-group&rs-pr=10000_100000&rs-er=0_1000&rs-ld=0_1000&rs-ac=2_23&rs-dcfc=0_400&rs-ub=10_200&rs-tw=0_3000&rs-ef=100_350&rs-sa=-1_5&rs-w=1000_3500&rs-c=0_5000&rs-y=2010_2030&s=1&p=0-200'

driver.get(url)

time.sleep(10)

html = driver.page_source

soup = BeautifulSoup(html, "html.parser")

boxes = soup.find_all('div', {'class':'list-item'})

for box in boxes:
    try:
        car_model = box.find('span',{'class':'hidden'}).get_text()
        model.append(car_model)
    except:
        model.append(None)
    try:
        efficience = box.find('span',{'class':'efficiency'}).get_text()
        efficiency.append(efficience)
    except:
        efficiency.append(None)
    try:
        range_value = box.find('span',{'class':'erange_real'}).get_text()
        range_km.append(range_value)
    except:
        range_km.append(None)
    try:
        batre = box.find('span',{'class':'battery_p'}).get_text()
        battery.append(batre)
    except:
        battery.append(None)
    try:
        price_tag = box.find('span',{'class':'country_uk'}).get_text()
        price.append(price_tag)
    except:
        price.append(None)
    try:
        available = box.find('div',{'class':'availability current'})
        if available is not None:
            availability.append('Available')
        else: 
            raise Exception
    except:
        try:
            available = box.find('div',{'class':'availability archive'})
            if available is not None:
                availability.append('Discontinued')
            else: 
                raise Exception
        except:
            availability.append(None)




driver.quit()

Syntax di bawah digunakan untuk menyimpan hasil webscraping ke pandas DataFrame.

In [27]:
df = pd.DataFrame(
    {
        'Car Model':model,
        'Efficiency (Wh/km)':efficiency,
        'Range (km)':range_km,
        'Battery (kWh)':battery,
        'Price (GBP)':price,
        'Availability':availability
        
        
    })

df.head()

,Car Model,Efficiency (Wh/km),Range (km),Battery (kWh),Price (GBP),Availability
0,Tesla Model 3 RWD,133 Wh/km,450 km,60.0 kWh,"£37,990",Available
1,MG MG4 Electric 64 kWh,171 Wh/km,360 km,61.7 kWh,"£29,745",Discontinued
2,BMW iX xDrive40,197 Wh/km,360 km,71.0 kWh,"£69,905",Discontinued
3,CUPRA Born 150 kW - 58 kWh,166 Wh/km,350 km,58.0 kWh,"£34,125",Discontinued
4,Fiat 500e Hatchback 42 kWh,159 Wh/km,235 km,37.3 kWh,"£20,245",Available


Syntax dibawah digunakan untuk menyimpan hasil Webscraping ke dalam file .csv

In [ ]:
df.to_csv('raw_data_EV_Sustainability.csv')

### B. Transform

Syntax dibawah digunakan untuk read file raw_data_EV_Sustainability.csv serta juga menyimpannya ke dalam variable df_ev.

In [28]:
df_ev = pd.read_csv('raw_data_EV_Sustainability.csv', index_col=0)

Syntax dibawah digunakan untuk menampilkan DataFrame df_ev yang saat ini masih dalam kondisi raw data.

In [29]:
df_ev

,Car Model,Efficiency (Wh/km),Range (km),Battery (kWh),Price (GBP),Availability
0,Tesla Model 3 RWD,133 Wh/km,450 km,60.0 kWh,"£37,990",Available
1,MG MG4 Electric 64 kWh,171 Wh/km,360 km,61.7 kWh,"£29,745",Discontinued
2,BMW iX xDrive40,197 Wh/km,360 km,71.0 kWh,"£69,905",Discontinued
3,CUPRA Born 150 kW - 58 kWh,166 Wh/km,350 km,58.0 kWh,"£34,125",Discontinued
4,Fiat 500e Hatchback 42 kWh,159 Wh/km,235 km,37.3 kWh,"£20,245",Available
...,...,...,...,...,...,...
195,Audi A6 Avant e-tron performance,165 Wh/km,575 km,94.9 kWh,"£71,740",Available
196,XPENG G6 AWD Performance,182 Wh/km,440 km,80.0 kWh,"£49,990",Available
197,Audi Q4 e-tron 45,183 Wh/km,420 km,77.0 kWh,"£51,360",Discontinued
198,Zeekr 7GT Core RWD,168 Wh/km,435 km,73.0 kWh,NaN,Available


Syntax di bawah digunakan untuk menampilkan 20 data pertama yang berada dalam df_ev (masih raw data).

In [30]:
df_ev.head(20)

,Car Model,Efficiency (Wh/km),Range (km),Battery (kWh),Price (GBP),Availability
0,Tesla Model 3 RWD,133 Wh/km,450 km,60.0 kWh,"£37,990",Available
1,MG MG4 Electric 64 kWh,171 Wh/km,360 km,61.7 kWh,"£29,745",Discontinued
2,BMW iX xDrive40,197 Wh/km,360 km,71.0 kWh,"£69,905",Discontinued
3,CUPRA Born 150 kW - 58 kWh,166 Wh/km,350 km,58.0 kWh,"£34,125",Discontinued
4,Fiat 500e Hatchback 42 kWh,159 Wh/km,235 km,37.3 kWh,"£20,245",Available
5,XPENG L03 RWD Long Range,165 Wh/km,420 km,69.5 kWh,NaN,Available
6,CUPRA Born 170 kW - 77 kWh,171 Wh/km,450 km,77.0 kWh,"£39,625",Discontinued
7,BMW i3 50 xDrive,152 Wh/km,715 km,108.7 kWh,"£53,005",Available
8,BMW iX3 50 xDrive,169 Wh/km,645 km,108.7 kWh,"£58,755",Available
9,Mercedes-Benz CLA 250+,145 Wh/km,585 km,85.0 kWh,"£43,250",Available


Syntax di bawah digunakan untuk menampilkan informasi rangkuman data df_ev.

Ringkasan df_ev.info() (raw data):

Berdasarkan output dibawah, dataset df_ev terdiri dari 200 rows dan 6 kolom, dimana sebesarnya memilki none/null values yang minimal. Hanya pada kolum 'Price (GBP)' dan 'Availability' yang memilki none/null values, dimana:
- 'Price (GBP)' memilki 32 none/null values
- 'Availability' memilki 3 none/null values

Keadaan none values ini di karenakan informasi tersebut memang tidak ada pada sumber, biasanya terjadi pada model yang belum resmi dirilis. None values ini dipertahankan dalam dataframe supaya tidak menghilangkan data valid pada kolum lain di baris yang sama.

Selain itu, saat ini semua 6 kolum merupakan data type 'str' atau string. Keseluruhan DataFrame ini menggunakan 9.5 KB memory, ini bisa di bilang termasuk dataset kecil.

In [31]:
df_ev.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Car Model           200 non-null    str  
 1   Efficiency (Wh/km)  200 non-null    str  
 2   Range (km)          200 non-null    str  
 3   Battery (kWh)       200 non-null    str  
 4   Price (GBP)         168 non-null    str  
 5   Availability        197 non-null    str  
dtypes: str(6)
memory usage: 9.5 KB


Syntax dibawah digunakan untuk mengubah data dalam kolom 'Efficiency (Wh/km)' untuk hanya memiliki angka dan mengubah data typenya menjadi Float.

In [32]:
df_ev['Efficiency (Wh/km)'] = df_ev['Efficiency (Wh/km)'].str.replace('Wh/km','')
df_ev['Efficiency (Wh/km)'] = df_ev['Efficiency (Wh/km)'].astype(float)

Syntax dibawah digunakan untuk mengubah data dalam kolom 'Range (km)' untuk hanya memiliki angka dan mengubah data typenya menjadi Float.

In [33]:
df_ev['Range (km)'] = df_ev['Range (km)'].str.replace('km','')
df_ev['Range (km)'] = df_ev['Range (km)'].astype(float)

Syntax dibawah digunakan untuk mengubah data dalam kolom 'Battery (kWh)' untuk hanya memiliki angka dan mengubah data typenya menjadi Float.

In [34]:
df_ev['Battery (kWh)'] = df_ev['Battery (kWh)'].str.replace('kWh','')
df_ev['Battery (kWh)'] = df_ev['Battery (kWh)'].astype(float)

Syntax dibawah digunakan untuk mengubah data dalam kolom 'Price (GBP)' untuk hanya memiliki angka dan mengubah data typenya menjadi Float.

In [38]:
df_ev['Price (GBP)'] = df_ev['Price (GBP)'].str.replace('£','')
df_ev['Price (GBP)'] = df_ev['Price (GBP)'].str.replace(',','')
df_ev['Price (GBP)'] = df_ev['Price (GBP)'].str.replace('*','')
df_ev['Price (GBP)'] = df_ev['Price (GBP)'].astype(float)

Syntax di bawah di gunakan untuk menampilkan 20 row pertama pada DataFrame df_ev setelah dibersihkan (clean data).

In [42]:
df_ev.head(20)

,Car Model,Efficiency (Wh/km),Range (km),Battery (kWh),Price (GBP),Availability
0,Tesla Model 3 RWD,133.0,450.0,60.0,37990.0,Available
1,MG MG4 Electric 64 kWh,171.0,360.0,61.7,29745.0,Discontinued
2,BMW iX xDrive40,197.0,360.0,71.0,69905.0,Discontinued
3,CUPRA Born 150 kW - 58 kWh,166.0,350.0,58.0,34125.0,Discontinued
4,Fiat 500e Hatchback 42 kWh,159.0,235.0,37.3,20245.0,Available
5,XPENG L03 RWD Long Range,165.0,420.0,69.5,NaN,Available
6,CUPRA Born 170 kW - 77 kWh,171.0,450.0,77.0,39625.0,Discontinued
7,BMW i3 50 xDrive,152.0,715.0,108.7,53005.0,Available
8,BMW iX3 50 xDrive,169.0,645.0,108.7,58755.0,Available
9,Mercedes-Benz CLA 250+,145.0,585.0,85.0,43250.0,Available


Syntax dibawah digunakan untuk menampilkan df_ev.info setelah dataframe telah dibersihkan.

Ringkasan df_ev.info() (clean data):

Berdasarkan output dibawah, setelah dibersihkan, kolum dengan data type 'str' atau string hanya 'Car Model' dan 'Availability. Kolum lainnya merupakan data 'float64'. None values tetap dimasuki dalam dataframe untuk mempertahankan data valid pada kolum lain di baris yang sama.

In [43]:
df_ev.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Car Model           200 non-null    str    
 1   Efficiency (Wh/km)  200 non-null    float64
 2   Range (km)          200 non-null    float64
 3   Battery (kWh)       200 non-null    float64
 4   Price (GBP)         168 non-null    float64
 5   Availability        197 non-null    str    
dtypes: float64(4), str(2)
memory usage: 9.5 KB


Syntax dibawah digunakan untuk menampilkan ringkasan statistik termasuk count, mean, standard deviation, minimum, maximum, dan quartile pada kolum-kolum numerik. Tujuan ini dilakukan adalah untuk memastikan hasil clean data sudah benar.

Berdasarkan output dibawah, ringkasan statistik menunjukkan values yang masuk akal pada setiap kolum numerik, sehingga bisa disimpulkan bahwa pembersihan data sudah dilakukan dengan benar.

In [44]:
df_ev.describe()

,Efficiency (Wh/km),Range (km),Battery (kWh),Price (GBP)
count,200.000000,200.000000,200.000000,168.000000
mean,172.455000,410.075000,70.881500,43848.029762
std,20.317034,105.950445,20.183087,20078.560377
min,130.000000,165.000000,24.000000,11990.000000
25%,160.000000,343.750000,58.000000,31926.250000
50%,169.000000,410.000000,74.400000,40240.000000
75%,182.000000,471.250000,80.150000,51721.250000
max,294.000000,720.000000,141.000000,180860.000000


Syntax dibawah digunakan untuk menyimpan df_ev yang telah dibersihkan (clean data) ke dalam file .csv.

In [45]:
df_ev.to_csv('clean_data_EV_Sustainability.csv', index = False)